# VCU IMU Basic Metrics

Loads one decoded `LOG####_imu.csv` file and prints simple IMU metrics for that run.

## Imports

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## Choose Log

In [2]:
# Directory containing decoded LOG####_imu.csv files.
# This works when the notebook is opened from either the repo root or the VCU folder.
DATA_DIR = Path("June11thTestingDataLog_AfterChargingBattery")

# Accepts 10, "10", "0010", or "LOG0010".
LOG_NUMBER = "LOG0010"

## Load IMU Data

In [3]:
ACCEL_COLS = ["accel_x_g", "accel_y_g", "accel_z_g"]
GYRO_COLS = ["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]
REQUIRED_COLS = ["t_ms", *ACCEL_COLS, *GYRO_COLS]
LOG_ID_RE = re.compile(r"^LOG(?P<num>\d+)$", re.IGNORECASE)


def normalize_log_id(log_number):
    text = str(log_number).strip()
    match = LOG_ID_RE.match(text)
    if match:
        return f"LOG{int(match.group('num')):04d}"
    if text.isdigit():
        return f"LOG{int(text):04d}"
    raise ValueError(f"Could not parse LOG number from {log_number!r}")


def resolve_data_dir(data_dir):
    data_dir = Path(data_dir).expanduser()
    candidates = [data_dir, Path.cwd() / data_dir, Path.cwd() / "VCU" / data_dir]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find data directory: {data_dir}")


def load_imu_log(data_dir, log_number):
    log_id = normalize_log_id(log_number)
    data_dir = resolve_data_dir(data_dir)
    path = data_dir / f"{log_id}_imu.csv"
    if not path.exists():
        raise FileNotFoundError(f"Could not find IMU log: {path}")

    raw = pd.read_csv(path)
    missing = [column for column in REQUIRED_COLS if column not in raw.columns]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    imu = raw.copy()
    for column in [*REQUIRED_COLS, "imu_temp_c"]:
        if column in imu.columns:
            imu[column] = pd.to_numeric(imu[column], errors="coerce")

    imu = imu.dropna(subset=REQUIRED_COLS).sort_values("t_ms").drop_duplicates("t_ms")
    imu = imu.reset_index(drop=True)
    imu.insert(1, "elapsed_s", (imu["t_ms"] - imu["t_ms"].iloc[0]) / 1000.0)
    return log_id, path, raw, imu


log_id, imu_path, raw_imu, imu = load_imu_log(DATA_DIR, LOG_NUMBER)

print(f"Loaded {imu_path}")
print(f"Rows loaded: {len(raw_imu):,}")
print(f"Rows after numeric cleanup: {len(imu):,}")
display(imu.head())

Loaded C:\Users\Nathan\Documents\Formula_E\BERT2026\BERT26-Firmware\BERT26-Firmware\VCU\June11thTestingDataLog_AfterChargingBattery\LOG0010_imu.csv
Rows loaded: 18,897
Rows after numeric cleanup: 18,897


,t_ms,elapsed_s,accel_x_g,accel_y_g,accel_z_g,gyro_x_dps,gyro_y_dps,gyro_z_dps,imu_temp_c
0,3280,0.00,0.2004,-0.0433,0.8828,-6.11,8.05,0.02,23.00
1,3290,0.01,0.2136,-0.0348,0.8798,-6.03,7.33,-0.03,23.00
2,3300,0.02,0.2021,-0.0403,0.8848,-6.08,6.75,0.00,23.02
3,3310,0.03,0.2231,-0.0409,0.9000,-6.14,8.70,-0.02,23.02
4,3320,0.04,0.2224,-0.0375,0.8947,-6.17,7.18,-0.09,23.02


## Basic Metrics

In [4]:
def basic_stats(df, columns):
    return df[columns].agg(["mean", "std", "min", "max"]).T


def magnitude_stats(df, columns, name):
    magnitude = pd.Series(np.linalg.norm(df[columns].to_numpy(dtype=float), axis=1), name=name)
    return magnitude.agg(["mean", "std", "min", "max"])


elapsed_s = imu["elapsed_s"]
dt_s = imu["t_ms"].diff().dropna() / 1000.0
median_dt_s = dt_s.median() if len(dt_s) else np.nan
sample_rate_hz = 1.0 / median_dt_s if median_dt_s > 0 else np.nan

timing = pd.Series(
    {
        "start_t_ms": imu["t_ms"].iloc[0],
        "end_t_ms": imu["t_ms"].iloc[-1],
        "duration_s": elapsed_s.iloc[-1],
        "samples": len(imu),
        "median_dt_s": median_dt_s,
        "sample_rate_hz": sample_rate_hz,
    }
)

print(f"{log_id} timing")
print(timing.to_string(float_format=lambda value: f"{value:.3f}"))

print("\nAcceleration by axis (g)")
display(basic_stats(imu, ACCEL_COLS))

print("Acceleration magnitude (g)")
print(magnitude_stats(imu, ACCEL_COLS, "accel_mag_g").to_string(float_format=lambda value: f"{value:.6f}"))

print("\nGyro by axis (deg/s)")
display(basic_stats(imu, GYRO_COLS))

print("Gyro magnitude (deg/s)")
print(magnitude_stats(imu, GYRO_COLS, "gyro_mag_dps").to_string(float_format=lambda value: f"{value:.6f}"))

if "imu_temp_c" in imu.columns:
    print("\nIMU temperature (deg C)")
    print(imu["imu_temp_c"].agg(["mean", "std", "min", "max"]).to_string(float_format=lambda value: f"{value:.3f}"))

LOG0010 timing
start_t_ms         3280.000
end_t_ms         202375.000
duration_s          199.095
samples           18897.000
median_dt_s           0.010
sample_rate_hz      100.000

Acceleration by axis (g)


,mean,std,min,max
accel_x_g,0.201356,0.142603,-1.1588,1.1929
accel_y_g,-0.087372,0.264444,-2.5967,2.5555
accel_z_g,0.887690,0.142735,-0.4935,3.9999


Acceleration magnitude (g)
mean   0.957775
std    0.171678
min    0.192405
max    4.103317

Gyro by axis (deg/s)


,mean,std,min,max
gyro_x_dps,-5.914666,2.276949,-24.46,34.63
gyro_y_dps,7.752870,7.365514,-100.99,76.55
gyro_z_dps,2.211561,13.130737,-117.31,104.67


Gyro magnitude (deg/s)
mean    14.568808
std     10.935113
min      0.682495
max    126.745716

IMU temperature (deg C)
mean   24.747
std     0.790
min    23.000
max    25.990


## Last Samples

In [5]:
display(imu.tail())

,t_ms,elapsed_s,accel_x_g,accel_y_g,accel_z_g,gyro_x_dps,gyro_y_dps,gyro_z_dps,imu_temp_c
18892,202335,199.055,0.1980,-0.0173,0.8838,-6.32,8.29,-0.08,25.97
18893,202345,199.065,0.2017,-0.0182,0.8840,-6.26,6.55,-0.05,25.96
18894,202355,199.075,0.1718,-0.0232,0.9015,-6.34,7.85,-0.06,25.97
18895,202365,199.085,0.1844,-0.0188,0.8881,-6.32,8.96,-0.06,25.98
18896,202375,199.095,0.1959,-0.0153,0.8776,-6.32,7.34,0.00,25.94


## Normalize data

In [7]:
# Calculate means for each sensor group
accel_means = imu[ACCEL_COLS].mean()
gyro_means = imu[GYRO_COLS].mean()

# Add normalized columns directly to the imu DataFrame
# Subtract means and assign to new columns with _norm suffix
imu[[f"{col}_norm" for col in ACCEL_COLS]] = imu[ACCEL_COLS] - accel_means
imu[[f"{col}_norm" for col in GYRO_COLS]] = imu[GYRO_COLS] - gyro_means
imu.tail()

,t_ms,elapsed_s,accel_x_g,accel_y_g,accel_z_g,gyro_x_dps,gyro_y_dps,gyro_z_dps,imu_temp_c,accel_x_g_norm,accel_y_g_norm,accel_z_g_norm,gyro_x_dps_norm,gyro_y_dps_norm,gyro_z_dps_norm
18892,202335,199.055,0.1980,-0.0173,0.8838,-6.32,8.29,-0.08,25.97,-0.003356,0.070072,-0.00389,-0.405334,0.53713,-2.291561
18893,202345,199.065,0.2017,-0.0182,0.8840,-6.26,6.55,-0.05,25.96,0.000344,0.069172,-0.00369,-0.345334,-1.20287,-2.261561
18894,202355,199.075,0.1718,-0.0232,0.9015,-6.34,7.85,-0.06,25.97,-0.029556,0.064172,0.01381,-0.425334,0.09713,-2.271561
18895,202365,199.085,0.1844,-0.0188,0.8881,-6.32,8.96,-0.06,25.98,-0.016956,0.068572,0.00041,-0.405334,1.20713,-2.271561
18896,202375,199.095,0.1959,-0.0153,0.8776,-6.32,7.34,0.00,25.94,-0.005456,0.072072,-0.01009,-0.405334,-0.41287,-2.211561
